<a href="https://colab.research.google.com/github/shoh0806/1-HTML-/blob/master/SASRec_BCE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#0. File upload


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving ratings.csv to ratings.csv


#1. import

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch import optim
import math
import random
import time
import os

#2 SEED 고정

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Device: {device}")

Device: cuda


#3. 하이퍼파라미터

In [ ]:
DATA_DIR     = "/content"
max_len      = 100
hidden_dim   = 100
num_heads    = 4
num_layers   = 4
batch_size   = 128
LR           = 0.001
weight_decay = 0.0001
epochs       = 30
stride       = 10
NEG_EVAL     = 100   # 평가용 negative 수
patience     = 10

#4. 데이터 전처리 및 데이터 분리

In [ ]:
ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))
ratings = ratings.sort_values(by=["userId", "timestamp"])

user_seq = ratings.groupby("userId")["movieId"].apply(list)

item_set  = ratings["movieId"].unique()
item2idx  = {item: i+1 for i, item in enumerate(item_set)}  # 1-based, 0은 padding
idx2item  = {i: item for item, i in item2idx.items()}

lengths = user_seq.apply(len)
print(f"평균: {lengths.mean():.1f}  최대: {lengths.max()}  최소: {lengths.min()}  중앙값: {lengths.median()}")

# 유저별 시퀀스를 idx로 변환
user_sequences = [[item2idx[i] for i in seq] for seq in user_seq]


# Train / Val / Test 분리 (leave-two-out)
train_sequences = []
val_data        = []
test_data       = []

for seq in user_sequences:
    if len(seq) < 20:
        continue
    train = seq[:-2]
    val   = seq[-2]
    test  = seq[-1]

    train_sequences.append(train)
    val_data.append((train, [val]))
    test_data.append((train + [val], [test]))

print(f"학습 유저 수: {len(train_sequences):,}")

평균: 165.6  최대: 2314  최소: 20  중앙값: 96.0
학습 유저 수: 6,040


#5. Dataset


In [ ]:
class SASRecDataset(Dataset):
    def __init__(self, sequences, max_len, stride):
        self.data     = []
        self.max_len  = max_len
        # 전체 아이템 집합 (negative sampling용)
        all_items = set()
        for seq in sequences:
            all_items.update(seq)
        self.all_items = list(all_items)  # 루프 밖에서 한 번만


        for seq in sequences:
            for i in range(1, len(seq), stride):
                window = seq[max(0, i - max_len): i + 1]

                input_seq = window[:-1]
                target    = window[1:]

                pad_len   = max_len - len(input_seq)
                input_seq = [0] * pad_len + input_seq
                target    = [0] * pad_len + target

                # 위치별 negative 1개씩 샘플링
                seq_set = set(seq)
                neg = []
                for t in target:
                    if t == 0:
                        neg.append(0)
                    else:
                        n = random.choice(self.all_items)
                        while n in seq_set:          # 이미 본 아이템 제외
                            n = random.choice(self.all_items)
                        neg.append(n)                # while 밖에서 append

                self.data.append((input_seq, target, neg))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_seq, target, neg = self.data[idx]
        return (torch.LongTensor(input_seq),
                torch.LongTensor(target),
                torch.LongTensor(neg))


dataset = SASRecDataset(train_sequences, max_len, stride)
loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)
print(f"학습 샘플 수: {len(dataset):,}")

학습 샘플 수: 100,899


#6. Model

In [ ]:
class SASRecBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=0.2,
            batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.ReLU(),
            nn.Linear(hidden_dim * 4, hidden_dim)
        )
        self.norm1   = nn.LayerNorm(hidden_dim)
        self.norm2   = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x, attn_mask, padding_mask):
        attn_output, attn_weights = self.attn(
            x, x, x,
            attn_mask=attn_mask,
            key_padding_mask=padding_mask,
            need_weights=True,
            average_attn_weights=False
        )
        attn_output = self.dropout(attn_output)
        x = self.norm1(x + attn_output)

        ffn_output = self.ffn(x)
        ffn_output = self.dropout(ffn_output)
        x = self.norm2(x + ffn_output)

        return x, attn_weights


class SASRec(nn.Module):
    def __init__(self, num_items, hidden_dim, max_len, num_heads, num_layers):
        super().__init__()
        self.item_emb = nn.Embedding(num_items + 1, hidden_dim, padding_idx=0)
        self.pos_emb  = nn.Embedding(max_len, hidden_dim)
        self.layers   = nn.ModuleList([
            SASRecBlock(hidden_dim, num_heads) for _ in range(num_layers)
        ])
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout    = nn.Dropout(0.2)

    def forward(self, batch_input):
        batch_size, seq_len = batch_input.size()

        pos = torch.arange(seq_len, device=batch_input.device).unsqueeze(0).repeat(batch_size, 1)

        x = self.item_emb(batch_input) * math.sqrt(self.item_emb.embedding_dim)
        x = x + self.pos_emb(pos)
        x = self.dropout(x)

        # Causal mask
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1)
        mask = mask.masked_fill(mask == 1, -1e9).float()

        # Padding mask
        padding_mask = (batch_input == 0)
        x = x.masked_fill(padding_mask.unsqueeze(-1), 0)

        attn_weights_all = []
        for layer in self.layers:
            x, attn_weights = layer(x, mask, padding_mask)  #SASRecBlock.forward도 자동 실행됨
            attn_weights_all.append(attn_weights)

        x = self.layer_norm(x)
        return x, attn_weights_all


model = SASRec(len(item2idx), hidden_dim, max_len, num_heads, num_layers).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=weight_decay)


#7. BCE Loss 함수 정의

In [ ]:
def bce_loss(model, batch_input, batch_target, batch_neg):
    output, _ = model(batch_input)               # (B, L, D) 여기서 SASRec forward 자동 호출된다.

    pos_emb = model.item_emb(batch_target)       # (B, L, D)
    neg_emb = model.item_emb(batch_neg)          # (B, L, D)

    pos_logits = (output * pos_emb).sum(-1)      # (B, L)
    neg_logits = (output * neg_emb).sum(-1)      # (B, L)

    mask = (batch_target != 0).float()           # padding 위치 무시

    loss = (
        F.binary_cross_entropy_with_logits(pos_logits, torch.ones_like(pos_logits),  reduction='none') +
        F.binary_cross_entropy_with_logits(neg_logits, torch.zeros_like(neg_logits), reduction='none')
    )
    return (loss * mask).sum() / mask.sum()

#8. 평가함수

In [ ]:
all_item_list = list(item2idx.values())

def evaluate(model, data, top_k=10):
    model.eval()
    HR, NDCG = 0, 0

    with torch.no_grad():
        for seq, gt in data:
            gt_idx  = gt[0]
            seq_set = set(seq)

            # negative 100개 샘플 (정답 및 이미 본 아이템 제외)
            negs = []
            while len(negs) < NEG_EVAL:
                n = random.choice(all_item_list)
                if n not in seq_set and n != gt_idx:
                    negs.append(n)

            candidates = [gt_idx] + negs         # 101개, 정답은 index 0

            # 패딩
            s   = seq[-max_len:]
            s   = [0] * (max_len - len(s)) + s
            inp = torch.LongTensor([s]).to(device)
            cand = torch.LongTensor([candidates]).to(device)

            output, _ = model(inp)
            h_last   = output[:, -1, :]                          # (1, D)
            cand_emb = model.item_emb(cand)                      # (1, 101, D)
            scores   = torch.bmm(cand_emb, h_last.unsqueeze(-1)).squeeze()  # (101,)

            rank = (scores[1:] > scores[0]).sum().item()         # 정답보다 높은 neg 수

            if rank < top_k:
                HR   += 1
                NDCG += 1 / math.log2(rank + 2)

    n = len(data)
    return HR / n, NDCG / n

#9. 학습

In [ ]:
best_val_ndcg = 0.0
best_model    = model.state_dict()
counter       = 0

print("\n══════ SASRec 학습 시작 ══════")
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for batch_input, batch_target, batch_neg in loader:
        batch_input  = batch_input.to(device)
        batch_target = batch_target.to(device)
        batch_neg    = batch_neg.to(device)

        optimizer.zero_grad()
        loss = bce_loss(model, batch_input, batch_target, batch_neg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) #gradient가 너무 커지는 걸 막는 cliping
        optimizer.step()
        total_loss += loss.item()

    elapsed  = time.time() - t0
    avg_loss = total_loss / len(loader)

    val_hr, val_ndcg = evaluate(model, val_data)
    print(f"Epoch {epoch:3d}/{epochs}  Loss: {avg_loss:.4f}  "
          f"HR@10: {val_hr:.4f}  NDCG@10: {val_ndcg:.4f}  ({elapsed:.1f}s)")

    # Early stopping
    if val_ndcg > best_val_ndcg:
        best_val_ndcg = val_ndcg
        counter       = 0
        best_model    = model.state_dict()
        print(f"  ★ Best model saved (epoch {epoch})")
    else:
        counter += 1
    if counter >= patience:
        print("Early stopping!")
        break



══════ SASRec 학습 시작 ══════


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Epoch   1/30  Loss: 1.5463  HR@10: 0.4298  NDCG@10: 0.2341  (35.3s)
  ★ Best model saved (epoch 1)
Epoch   2/30  Loss: 0.9574  HR@10: 0.4434  NDCG@10: 0.2401  (35.6s)
  ★ Best model saved (epoch 2)
Epoch   3/30  Loss: 0.8303  HR@10: 0.6060  NDCG@10: 0.3416  (37.0s)
  ★ Best model saved (epoch 3)
Epoch   4/30  Loss: 0.6529  HR@10: 0.7374  NDCG@10: 0.4566  (36.7s)
  ★ Best model saved (epoch 4)
Epoch   5/30  Loss: 0.5309  HR@10: 0.8030  NDCG@10: 0.5427  (36.7s)
  ★ Best model saved (epoch 5)
Epoch   6/30  Loss: 0.4639  HR@10: 0.8341  NDCG@10: 0.5819  (36.6s)
  ★ Best model saved (epoch 6)
Epoch   7/30  Loss: 0.4299  HR@10: 0.8416  NDCG@10: 0.5890  (36.6s)
  ★ Best model saved (epoch 7)
Epoch   8/30  Loss: 0.4110  HR@10: 0.8421  NDCG@10: 0.6009  (36.6s)
  ★ Best model saved (epoch 8)
Epoch   9/30  Loss: 0.3984  HR@10: 0.8500  NDCG@10: 0.6096  (36.5s)
  ★ Best model saved (epoch 9)
Epoch  10/30  Loss: 0.3899  HR@10: 0.8493  NDCG@10: 0.6072  (36.7s)
Epoch  11/30  Loss: 0.3842  HR@10: 0.8467

In [ ]:
print("\n──── 최종 테스트 평가 ────")
model.load_state_dict(best_model)
test_hr, test_ndcg = evaluate(model, test_data)
print(f"Test HR@10:   {test_hr:.4f}")
print(f"Test NDCG@10: {test_ndcg:.4f}")



──── 최종 테스트 평가 ────
Test HR@10:   0.8275
Test NDCG@10: 0.5963
